# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/singhmahip688-hue/flyrank-ml-internhip/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os

REPO_DIR = "flyrank-ml-internhip"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/singhmahip688-hue/flyrank-ml-internhip.git

os.chdir(REPO_DIR)
print("Now working in:", os.getcwd())
!ls

Now working in: /content/flyrank-ml-internhip
AGENTS.md  DATA_USE.md	LICENSE    README.md	     SETUP.md	 work
CLAUDE.md  docs		notebooks  requirements.txt  skills
data	   GUIDE.md	outputs    scripts	     submission


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# ---------- Signal 1: staleness (behind the refresh flags) ----------
# freshness_tier already exists in the data — use it directly
bucket1 = df.groupby('freshness_tier', observed=True).agg(
    n=('content_id', 'size'),
    avg_ctr=('ctr', 'mean')
).sort_index()
print("Signal 1 — staleness vs CTR")
print(bucket1)
print()

# ---------- Signal 2: CTR vs position (behind the CTR-fix logic) ----------
# avg_position == 0 means "no data" — must exclude, not treat as position zero
valid_pos = df[df['avg_position'] > 0]

bucket2 = valid_pos.groupby('position_tier', observed=True).agg(
    n=('content_id', 'size'),
    avg_ctr=('ctr', 'mean'),
    avg_impressions=('impressions_90d', 'mean')  # volume floor, per the data dictionary warning
).sort_index()
print("Signal 2 — position vs CTR (volume floor shown)")
print(bucket2)

Signal 1 — staleness vs CTR
                    n   avg_ctr
freshness_tier                 
0-30            20480  0.609021
181+              174  3.693276
31-90             175  0.117543
91-180           9171  0.238367

Signal 2 — position vs CTR (volume floor shown)
                   n   avg_ctr  avg_impressions
position_tier                                  
deep            1319  0.150212       931.218347
page_1         11814  0.652467      7582.142966
page_3_5        7242  0.222484      4858.086302
striking        7304  0.323239      3147.871577
top_3           1116  2.764453      6299.900538


Rule (plain words):
Flag a page REFRESH if it's stale (freshness_tier 91-180 or 181+) and its CTR
is below the average CTR for its position tier. Flag QUICK_WIN if it has high
impressions_90d (impression_tier good/excellent) but below-average CTR for its
position. Otherwise MONITOR.

Signal 1 (staleness): [fill in after running — e.g. "CONFIRMED: avg_ctr drops
from X in 0-30 to Y in 181+"]

Signal 2 (CTR vs position): [fill in — e.g. "CONFIRMED: avg_ctr is highest in
top_3, drops through page_1/striking/page_3_5/deep — but top_3 has a low
volume floor (~53 avg impressions), so treat with caution per data dictionary"]

Reason codes this rule can output: STALE_LOW_CTR, HIGH_VOL_LOW_CTR, NO_STRONG_SIGNAL

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

# expected CTR = median CTR for that position tier (excluding no_data)
pos_median = valid_pos.groupby('position_tier', observed=True)['ctr'].median()

def expected_ctr(row):
    if row['avg_position'] <= 0 or row['position_tier'] == 'no_data':
        return None
    return pos_median.get(row['position_tier'], None)

df['expected_ctr'] = df.apply(expected_ctr, axis=1)
df['ctr_gap'] = df['expected_ctr'] - df['ctr']   # positive = underperforming for its position

# normalize signals to 0-1 for scoring
df['staleness_score'] = (df['days_since_last_update'] - df['days_since_last_update'].min()) / \
                         (df['days_since_last_update'].max() - df['days_since_last_update'].min())
df['volume_score'] = (df['impressions_90d'] - df['impressions_90d'].min()) / \
                      (df['impressions_90d'].max() - df['impressions_90d'].min())
df['ctr_gap_score'] = df['ctr_gap'].clip(lower=0)
df['ctr_gap_score'] = df['ctr_gap_score'] / df['ctr_gap_score'].max()

def assign_action(row):
    if pd.isna(row['ctr_gap_score']):
        return "MONITOR", "NO_STRONG_SIGNAL"
    if row['staleness_score'] > 0.5 and row['ctr_gap_score'] > 0.5:
        return "REFRESH", "STALE_LOW_CTR"
    elif row['volume_score'] > 0.5 and row['ctr_gap_score'] > 0.5:
        return "QUICK_WIN", "HIGH_VOL_LOW_CTR"
    else:
        return "MONITOR", "NO_STRONG_SIGNAL"

df['score'] = df[['staleness_score','ctr_gap_score']].fillna(0).mean(axis=1) * 0.7 + \
              df['volume_score'].fillna(0) * 0.3

df[['action', 'reason_code']] = df.apply(assign_action, axis=1, result_type='expand')

ranked = df.sort_values('score', ascending=False)

os.makedirs("work/outputs", exist_ok=True)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(ranked[['content_id','score','action','reason_code']].head(20))

                 content_id     score   action       reason_code
26242  content_55a5b1c46474  0.700020  REFRESH     STALE_LOW_CTR
24216  content_1b4ec72dafd4  0.699060  REFRESH     STALE_LOW_CTR
8631   content_e2b702f4f92b  0.663323  REFRESH     STALE_LOW_CTR
15608  content_06e19c6486b0  0.663312  REFRESH     STALE_LOW_CTR
21984  content_02b0d6e30129  0.643650  REFRESH     STALE_LOW_CTR
3723   content_f488400fca67  0.636111  REFRESH     STALE_LOW_CTR
1147   content_ab27c30d81f4  0.635140  REFRESH     STALE_LOW_CTR
29906  content_07ce98c6085a  0.635129  REFRESH     STALE_LOW_CTR
15052  content_7736e6144f3b  0.635086  REFRESH     STALE_LOW_CTR
24557  content_84d12054c0c0  0.635081  REFRESH     STALE_LOW_CTR
7222   content_8f2c815af658  0.632262  REFRESH     STALE_LOW_CTR
7445   content_c8e9d6ab9013  0.567831  MONITOR  NO_STRONG_SIGNAL
16417  content_f4b3081037b3  0.566398  REFRESH     STALE_LOW_CTR
15947  content_40e140ba2934  0.566398  REFRESH     STALE_LOW_CTR
5653   content_10b9f5f766

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = ranked.head(20)
print(top20[['content_id','score','action','reason_code',
             'days_since_last_update','ctr','avg_position',
             'impressions_90d','freshness_tier','position_tier']].to_string())


                 content_id     score   action       reason_code  days_since_last_update  ctr  avg_position  impressions_90d freshness_tier position_tier
26242  content_55a5b1c46474  0.700020  REFRESH     STALE_LOW_CTR                     373  0.0           7.5               35           181+        page_1
24216  content_1b4ec72dafd4  0.699060  REFRESH     STALE_LOW_CTR                     372  0.0           7.0                2           181+        page_1
8631   content_e2b702f4f92b  0.663323  REFRESH     STALE_LOW_CTR                     334  0.0           9.3               30           181+        page_1
15608  content_06e19c6486b0  0.663312  REFRESH     STALE_LOW_CTR                     334  0.0           5.0               10           181+        page_1
21984  content_02b0d6e30129  0.643650  REFRESH     STALE_LOW_CTR                     313  0.0           6.9              176           181+        page_1
3723   content_f488400fca67  0.636111  REFRESH     STALE_LOW_CTR            

Top-20 review

Note: every row in this top-20 has ctr = 0.0 — meaning the rule is currently
being dominated by "zero clicks" pages rather than a graded CTR gap. Impression
counts vary hugely (1 to 208,678), so confidence differs a lot row to row even
though they all got the same reason code. I flag this pattern properly in
Section 4.

1. content_55a5b1c46474 — REFRESH, STALE_LOW_CTR. Moderate confidence: 35
   impressions over 90d with 0 clicks, stale 373 days, page_1 position (7.5).
   Would be wrong if traffic is seasonal and this page normally converts in a
   different quarter.

2. content_1b4ec72dafd4 — REFRESH, STALE_LOW_CTR. Low confidence: only 2
   impressions — 0 clicks on 2 impressions tells us almost nothing. Would be
   wrong simply because the sample size is too small to trust.

3. content_e2b702f4f92b — REFRESH, STALE_LOW_CTR. Moderate confidence: 30
   impressions, 0 clicks, stale 334 days. Would be wrong if the query it ranks
   for is purely navigational (people don't click search results for it).

4. content_06e19c6486b0 — REFRESH, STALE_LOW_CTR. Low confidence: only 10
   impressions. Would be wrong if this is a low-intent long-tail keyword
   where 0 CTR is normal, not a content problem.

5. content_02b0d6e30129 — REFRESH, STALE_LOW_CTR. High confidence: 176
   impressions with 0 clicks despite page_1 position — real evidence of a
   problem, not noise. Would be wrong if the SERP snippet/title is fine but
   the topic itself has genuinely low click intent.

6. content_f488400fca67 — REFRESH, STALE_LOW_CTR. High confidence: 155
   impressions, 0 clicks, stale 305 days. Would be wrong if a competing
   internal page already captures the clicks for this query (cannibalization,
   not staleness).

7. content_ab27c30d81f4 — REFRESH, STALE_LOW_CTR. Moderate-high confidence:
   103 impressions, 0 clicks. Would be wrong if this page targets a branded
   query where users navigate directly instead of clicking search results.

8. content_07ce98c6085a — REFRESH, STALE_LOW_CTR. Moderate confidence: 85
   impressions, 0 clicks. Would be wrong if the ranking snippet metadata was
   recently improved but GSC data hasn't caught up yet.

9. content_7736e6144f3b — REFRESH, STALE_LOW_CTR. Low confidence: only 11
   impressions — too few to separate "bad content" from "just unlucky."

10. content_84d12054c0c0 — REFRESH, STALE_LOW_CTR. Very low confidence: 1
    impression total. This is essentially no data — should not be treated the
    same as row 5 or 6 despite having a similar score.

11. content_8f2c815af658 — REFRESH, STALE_LOW_CTR. Low confidence: 7
    impressions, 0 clicks — sample too small to act on with certainty.

12. content_c8e9d6ab9013 — MONITOR, NO_STRONG_SIGNAL. Interesting case: 208,678
    impressions with 0 clicks (or rounds to 0.0), only 104 days stale. This is
    the strongest possible evidence of a real CTR problem in the whole dataset,
    but it landed on MONITOR because staleness_score was below 0.5 (104 days
    isn't "181+"). This shows a weakness in my rule — I flag it in Section 4.

13. content_f4b3081037b3 — REFRESH, STALE_LOW_CTR. Very low confidence: 2
    impressions. Same issue as row 2 — near-zero sample size.

14. content_40e140ba2934 — REFRESH, STALE_LOW_CTR. Very low confidence: 2
    impressions, same reasoning as row 13.

15. content_10b9f5f766b4 — REFRESH, STALE_LOW_CTR. Low confidence: 15
    impressions, 0 clicks, stale 211 days.

16. content_3b7c76f80c79 — REFRESH, STALE_LOW_CTR. Low confidence: 13
    impressions — too few to be certain content quality is the cause.

17. content_94d6b610fb8e — REFRESH, STALE_LOW_CTR. Low confidence: 11
    impressions, same reasoning as row 16.

18. content_8bc10f396d2e — REFRESH, STALE_LOW_CTR. Low confidence: 8
    impressions, top position (3.6) yet 0 clicks — surprising for such a good
    position, but sample size is too small to trust fully.

19. content_1dcf67c62f50 — REFRESH, STALE_LOW_CTR. Low confidence: 7
    impressions, weak position (9.6) — plausible 0 CTR is just due to rank.

20. content_f783292bc4a0 — REFRESH, STALE_LOW_CTR. Low confidence: 6
    impressions — too small a sample to separate signal from noise.

Overall observation: high-impression, 0-CTR pages (rows 5, 6, 12) are the
most trustworthy REFRESH candidates. Most of the rest are low-impression rows
where 0 CTR could just be small-sample noise rather than a real content
problem — my rule doesn't currently distinguish between these two cases,
which is a real limitation to address in Section 4.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Weak picks: flag any top-20 rows with low impression counts (unstable CTR)
weak = top20[top20['impressions_90d'] < 100]
print("Weak picks (low impressions, CTR is noisy):")
print(weak[['content_id','impressions_90d','ctr','action']])

# Leakage check
used_cols = ['days_since_last_update', 'ctr', 'avg_position', 'impressions_90d',
             'freshness_tier', 'position_tier']
leakage_cols = ['trend_direction', 'trend_pct', 'is_declining_label']

overlap = [c for c in used_cols if c in leakage_cols]
print("\nColumns used in scoring:", used_cols)
print("Known leakage/label columns:", leakage_cols)
print("Overlap (should be empty):", overlap)

Weak picks (low impressions, CTR is noisy):
                 content_id  impressions_90d  ctr   action
26242  content_55a5b1c46474               35  0.0  REFRESH
24216  content_1b4ec72dafd4                2  0.0  REFRESH
8631   content_e2b702f4f92b               30  0.0  REFRESH
15608  content_06e19c6486b0               10  0.0  REFRESH
29906  content_07ce98c6085a               85  0.0  REFRESH
15052  content_7736e6144f3b               11  0.0  REFRESH
24557  content_84d12054c0c0                1  0.0  REFRESH
7222   content_8f2c815af658                7  0.0  REFRESH
16417  content_f4b3081037b3                2  0.0  REFRESH
15947  content_40e140ba2934                2  0.0  REFRESH
5653   content_10b9f5f766b4               15  0.0  REFRESH
4013   content_3b7c76f80c79               13  0.0  REFRESH
10190  content_94d6b610fb8e               11  0.0  REFRESH
19447  content_8bc10f396d2e                8  0.0  REFRESH
15589  content_1dcf67c62f50                7  0.0  REFRESH
7719   conte

Weak picks: [name 2-3 content_ids from the weak table] — flagged because
impressions_90d is under 100, so their CTR is a noisy estimate (one extra
click swings CTR a lot at this volume — per the data dictionary's volume
floor warning).

Leakage check: confirmed no overlap between used_cols and leakage_cols.
trend_direction and trend_pct were not used anywhere in the scoring — both
are label-source columns and excluded per the data dictionary.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.